In [1]:
library(here)

knitr::opts_chunk$set(echo = TRUE)
options(box.path = here())

box::use(
  r / load
)
box::reload(load)

library(tidyverse)
library(texreg)
library(ggeffects)
library(ggpubr)

library(lme4)
library(lmerTest)
library(marginaleffects)
library(sjstats)
library(slider)

theme_set(theme_minimal())


here() starts at /Users/lukas/git/ytpop
── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the ]8;;http://conflicted.r-lib.org/conflicted package]8;; to force all conflicts to become errors
Version:  1.39.4
Date:     2024-07-23
Author:   Philip Leifeld (University of Manchester)

Consider submitting praise using the praise or praise_interactive functions.
Please cite the JSS article in your publications -- see citation("texreg").

Attaching package: ‘texreg’

The following object is m

In [2]:
channels <- load$channels()
videos <- load$videos(filtered = TRUE)
sentences <- load$sentences(filtered = TRUE)
popbert <- load$popbert(filtered = TRUE)

df_sents <- sentences |>
  left_join(popbert, by = "sentence_id") |>
  group_by(video_id) |>
  summarize(
    n_sents = n(),
    elite = mean(elite),
    pplcentr = mean(pplcentr)
  ) 

In [3]:
centered <- function(x) (x - mean(x))
z_transform <- function(x) as.vector(scale(x))

df <- channels |>
  left_join(videos, by = "channel_id") |>
  left_join(df_sents, by = "video_id") |>
  mutate(
    channel = as_factor(channel)
  )

df <- df |> 
  # filter(video_datetime_upload > "2020-10-01") |> 
  mutate(
    released_at = video_datetime_upload |> lubridate::floor_date(unit="month") |> format("%Y-%m"),
    released_at = released_at |> factor(levels=sort(unique(released_at)), ordered=TRUE),
    released_year = video_datetime_upload |> lubridate::floor_date(unit="year") |> format("%Y"),
    released_year = released_year |> factor(levels=sort(unique(released_year)), ordered=FALSE)
  ) |> 
  filter(channel != "FDP") |>
  mutate(
    channel = channel |> fct_drop(),
    log_video_likes = log(video_likes),
    log_video_views = log(1 + video_views),
    likes_per_view = video_likes / video_views,
    log_video_duration = log(video_duration),
    log_n_sents = log(n_sents),
    # pplcentr = centered(pplcentr),
    elite = centered(elite)
  ) |> 
  arrange(channel, video_datetime_upload) |> 
  group_by(channel) |> 
  mutate(
    outlier = scale(video_views),
    channel_rolling_views = slider::slide_index_dbl(
      video_views,
      .i = video_datetime_upload,
      .f = ~mean(.x, na.rm=T),
      .before = lubridate::weeks(4)
    )
  ) |>
  ungroup()

count_before <- nrow(df)
df <- df |> 
  filter(
    between(outlier, -2.5, 2.5)
  )
count_after <- nrow(df)

print(paste("Removed", count_before - count_after, "outliers"))

[1] "Removed 205 outliers"


In [4]:
set.seed(100)

k_means_result <- kmeans(
  df |> select(log_n_sents, log_video_duration),
  centers = 3,
  algorithm = "Lloyd",
  iter.max=1000
)

df$cluster <- k_means_result$cluster |> factor()

cluster_mapping <- df |> 
  group_by(cluster) |> 
  summarize(
    mean_duration = mean(video_duration),
    .groups = "drop"
  ) |> 
  arrange(mean_duration)


df <- df |> 
  left_join(cluster_mapping, by = "cluster")

In [5]:
show_df <- df |>
  group_by(channel, cluster) |> 
  summarize(
    n = n(),
    max_elite = max(elite),
    elite = mean(elite),
    duration = mean(video_duration),
    n_sents = mean(n_sents),
    views = median(video_views),
    likes = median(video_likes, na.rm = T),
    .groups="drop"
  ) |> arrange(channel, duration)

print(show_df, n=50)

# A tibble: 21 × 9
   channel cluster     n max_elite   elite duration n_sents  views  likes
   <fct>   <fct>   <int>     <dbl>   <dbl>    <dbl>   <dbl>  <dbl>  <dbl>
 1 SPD     2         578   0.309   -0.0836     52.2    10.2  3208   117  
 2 SPD     3         237   0.160   -0.0898    362.     52.6  1162    41.5
 3 SPD     1         215   0.00540 -0.0974   2352.    349.   1507    45  
 4 CSU     2          74   0.214   -0.0876     72.4    12.1   680.   20.5
 5 CSU     3          69   0.130   -0.0822    226.     38.1  1134    27  
 6 CSU     1          25  -0.0627  -0.103    2147.    400.    819    18  
 7 CDU     2         442   0.380   -0.100      70.1    10.8  1222.   30  
 8 CDU     3         229   0.130   -0.107     327.     48.8   910    20  
 9 CDU     1         148  -0.0283  -0.102    2461.    391.   2678    33  
10 Greens  2         156   0.280   -0.0922     76.2    12.9  2426.  130. 
11 Greens  3         345   0.230   -0.0832    337.     49.9   867    19  
12 Greens  1       

In [285]:
options(repr.plot.width = 2, repr.plot.height = 0.75, repr.plot.res = 100)

In [6]:
library(caret)

ctrl <- trainControl(
  method = "cv",
  number = 5,
  savePredictions = "final"
)

tunegrid <- expand.grid(
  mtry = c(1:7),
  min.node.size = c(1, 2, 3, 4, 10, 20, 50, 80, 100),
  splitrule = "gini"
)

form <- as.formula(video_views ~ channel)

model <- train(
  form,
  method = "ranger",
  data = data$train,
  metric = "Accuracy",
  tuneGrid = tunegrid,
  trControl = ctrl,
  importance = "impurity",
  respect.unordered.factors = "partition"
)

: [1m[33mError[39m in `library()`:[22m
[33m![39m there is no package called ‘caret’

In [ ]:

ggplot(df, aes(x=log_video_duration, y=elite, color=cluster)) +
    geom_jitter(aes(size=video_views), alpha=0.8, width=0.01, height=0.01)


In [ ]:
ggplot(df, aes(x=video_datetime_upload, y=channel_rolling_views)) +
  geom_line() +
  facet_wrap(~channel, ncol=2, scales="free_y")

# Models


## Video Views

In [ ]:
ggplot(df, aes(elite)) +
  geom_histogram(bins=50)

In [ ]:
ggplot(df, aes(log_video_views)) +
  geom_histogram(bins=50)

In [ ]:
library(glmmTMB)

eq2 <- video_views ~ 1 + elite + released_year + elite:released_year + log(video_duration) + (1 + elite + log(video_duration) + released_year + elite:released_year | channel) 
model <- glmmTMB(eq2, data=df, family="poisson")
performance::check_overdispersion(model)

In [ ]:
model <- glmmTMB(eq2, data=df, family=nbinom2)
summary(model)

In [ ]:
sjPlot::plot_model(model, type="re")

In [ ]:
performance::icc(model)

In [ ]:
preds <- ggeffects::predict_response(
  model,
  terms = c("elite", "channel"),
  margin = "empirical"
) 

plot(preds, show_ci=F)
print(preds, n=100)

### Elite Models

In [ ]:
nullmodel <- lmerTest::lmer(video_views ~ 1 + (1 | channel), data=df, REML=F)
summary(nullmodel)

In [ ]:
performance::icc(nullmodel)

In [ ]:
model1 <- lmerTest::lmer(log_video_views ~ 1 + elite + cluster + (1 | channel), data=df, REML=F)
summary(model1)

In [ ]:
performance::icc(model1)

In [ ]:
model2 <- lmerTest::lmer(log_video_views ~ 1 + elite + cluster + released_year + elite:cluster + (1 + elite | channel), data=df, REML=F)
summary(model2)

In [ ]:
performance::icc(model2)

In [ ]:
sjPlot::plot_model(model2, type="eff", terms = c("elite", "cluster"))

In [ ]:
sjPlot::plot_model(model2, type="re", pred.type="fe")


In [ ]:
sjPlot::plot_model(model2, type="pred", terms = c("elite [-0.2, 0, 0.2]", "cluster"))

In [83]:
model <- lme4::lmer(
  formula = log_video_views ~ 1 + elite + (1 + elite | cluster) + (1 + elite | channel) + (1 | released_year) ,
  data = df
)

In [ ]:
summary(model)

In [ ]:
performance::icc(model)

In [ ]:
ggeffect(model)

In [ ]:
sjPlot::plot_model(model, type="re", pred.type="fe", terms=c("channel", "cluster"))

In [ ]:
preds <- ggpredict(
  model,
  terms = c("cluster", "elite"),
  type="random",
  margin="empirical"
)
plot(preds, show_ci = F) + xlim(c(0, 0.6))

## Pplcentr model

In [23]:
model <- lme4::lmer(
  formula = log_video_views ~ 1 + pplcentr + n_sents + released_at + (1 + pplcentr | channel),
  data = df
)

In [ ]:
performance::icc(model)

In [ ]:
sjPlot::plot_model(model, type="re")

In [ ]:
preds <- ggpredict(
  model,
  terms = c("pplcentr", "channel"),
  type="random",
  margin="empirical"
)
preds$predicted <- exp(preds$predicted)

plot(preds, show_ci = F)